# Clase 8 — Gastar menos combustible con datos
## 🟢 Nivel NOVATO — Optimización sin programar

**Curso:** IA Aplicada a la Producción Pesquera · UTN FRCh · PesquerosEnIA · 2026
**Autor:** Ariel Giamportone

---

**Para vos que nunca programaste.** Apretá ▶ y mirá el resultado.

**La idea:** el combustible es el mayor costo de una marea. Con datos se puede encontrar la
**velocidad más económica** y estimar **cuánta plata se ahorra** una flota entera.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)
print('✓ Librerías cargadas')

## 1. El combustible: el mayor costo de la marea

En un arrastrero de altura, el gasoil es el **25–40% del costo** de cada viaje. Ir más
rápido gasta **mucho** más (el consumo sube con el cubo de la velocidad), pero ir muy lento
también cuesta (más días de tripulación). Existe una **velocidad justa**.

In [ ]:
# ── Curva de costo total vs velocidad ─────────────────────────────────────────
velocidades = np.arange(5.5, 13.5, 0.1)  # nudos
distancia_caladero_km = 300  # km al caladero (ejemplo: Puerto Madryn → zona merluza)

# Conversión: 1 nudo = 1.852 km/h
velocidades_kmh = velocidades * 1.852
tiempo_viaje_h = distancia_caladero_km / velocidades_kmh  # horas de navegación

# Consumo: función cúbica de velocidad (ley de Admiralty)
# C(v) = k * v^3 * tiempo (simplificado)
k_consumo = 0.003  # constante para buque tipo arrastrero
consumo_viaje_tn = k_consumo * (velocidades ** 3) * (distancia_caladero_km / velocidades_kmh)

# Costo de combustible
precio_gasoil_usd_tn = 850
costo_combustible = consumo_viaje_tn * precio_gasoil_usd_tn

# Costo operativo del tiempo (tripulación + charter + costos fijos)
costo_por_hora_usd = 1_400  # USD/hora en navegación
costo_tiempo = tiempo_viaje_h * costo_por_hora_usd

# Costo total
costo_total = costo_combustible + costo_tiempo

velocidad_optima = velocidades[np.argmin(costo_total)]
print(f'Velocidad de mínimo consumo: {velocidades[np.argmin(consumo_viaje_tn)]:.1f} kn')
print(f'Velocidad de mínimo costo total: {velocidad_optima:.1f} kn')
print(f'Costo total mínimo por viaje (ida al caladero): USD {min(costo_total):,.0f}')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(velocidades, consumo_viaje_tn, 'steelblue', linewidth=2)
axes[0].axvline(velocidades[np.argmin(consumo_viaje_tn)], linestyle='--',
                color='gray', label=f'Mín. consumo: {velocidades[np.argmin(consumo_viaje_tn)]:.1f} kn')
axes[0].set_xlabel('Velocidad (nudos)')
axes[0].set_ylabel('Consumo de combustible (tn)')
axes[0].set_title('Consumo vs Velocidad (dist. 300 km)\nLey cúbica del consumo naval')
axes[0].legend()

axes[1].plot(velocidades, costo_total / 1000, 'darkorange', linewidth=2, label='Costo total')
axes[1].plot(velocidades, costo_combustible / 1000, 'steelblue', linestyle='--',
             linewidth=1.5, alpha=0.7, label='Solo combustible')
axes[1].plot(velocidades, costo_tiempo / 1000, 'green', linestyle='--',
             linewidth=1.5, alpha=0.7, label='Solo tiempo (tripulación)')
axes[1].axvline(velocidad_optima, linestyle='-', color='red', linewidth=2,
                label=f'Velocidad óptima: {velocidad_optima:.1f} kn')
axes[1].set_xlabel('Velocidad (nudos)')
axes[1].set_ylabel('Costo (miles de USD)')
axes[1].set_title('Costo total vs Velocidad\n(combustible + tiempo operativo)')
axes[1].legend(fontsize=9)

plt.suptitle('Optimización de Velocidad de Crucero — Arrastrero PCA (dist. 300 km)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 👀 Hay una velocidad óptima

La curva naranja (costo total) tiene un **punto más bajo**: esa es la velocidad que menos
plata gasta por viaje. Ni muy rápido ni muy lento.

## 2. ¿Cuánto se ahorra en toda la flota?

Si 20 barcos aplican esta optimización durante un año, el ahorro se acumula. Apretá ▶.

In [ ]:
# ── Simulación de impacto económico ───────────────────────────────────────────
np.random.seed(42)
n_barcos = 20
n_mareas_anio = 12
precio_gasoil = 850  # USD/tn

# Consumo sin optimización: promedio ~90 tn/marea, con variabilidad
consumo_sin_opt = np.random.normal(92, 12, (n_barcos, n_mareas_anio)).clip(50, 160)

# Con optimización: reducción del 12-18% (velocidad + rutas + mantenimiento)
factor_reduccion = np.random.uniform(0.82, 0.88, (n_barcos, n_mareas_anio))
consumo_con_opt = consumo_sin_opt * factor_reduccion

ahorro_tn = consumo_sin_opt - consumo_con_opt
ahorro_usd = ahorro_tn * precio_gasoil

print('=== Impacto económico de la optimización IA ===')
print(f'Flota: {n_barcos} arrastreros de altura')
print(f'Mareas por año: {n_mareas_anio}')
print()
print(f'Ahorro promedio por marea: {ahorro_tn.mean():.1f} tn de gasoil')
print(f'  = USD {ahorro_usd.mean():,.0f} por marea')
print()
print(f'Ahorro anual por barco: {ahorro_tn.mean() * n_mareas_anio:.0f} tn')
print(f'  = USD {ahorro_usd.mean() * n_mareas_anio:,.0f} por barco/año')
print()
print(f'AHORRO TOTAL FLOTA (20 barcos, 1 año): USD {ahorro_usd.sum():,.0f}')
print(f'  ≈ USD {ahorro_usd.sum() / 1e6:.1f} millones/año')

# Visualización
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_barcos)
ax.bar(x - 0.2, consumo_sin_opt.mean(axis=1), 0.38,
       label='Sin optimización', color='#EF5350', alpha=0.85, edgecolor='white')
ax.bar(x + 0.2, consumo_con_opt.mean(axis=1), 0.38,
       label='Con optimización IA', color='#42A5F5', alpha=0.85, edgecolor='white')
ax.set_xlabel('Embarcación (1–20)')
ax.set_ylabel('Consumo promedio por marea (tn gasoil)')
ax.set_title('Impacto de la Optimización IA en el Consumo de Combustible\n'
             f'Flota de {n_barcos} arrastreros — Ahorro total: USD {ahorro_usd.sum():,.0f}/año',
             fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 👀 Para tu empresa

El ahorro de combustible de una flota completa se mide en **millones de dólares al año**,
solo ajustando velocidad y rutas con datos. Y menos gasoil = menos emisiones.

## ✅ Qué te llevás
- El combustible es el costo que más se puede **optimizar**.
- Hay una **velocidad óptima** que minimiza el costo total.
- Con datos, el ahorro de una flota es **enorme** y además más sostenible.

Cuando quieras, pasá al **Nivel Intermedio**. Comunidad: github.com/PesquerosEnIA